# PIPELINE DE ANÁLISE SEMÂNTICA — PEEL PHASE 1

Este notebook implementa um pipeline de análise semântica e clustering lexical correspondente à Fase 1 do PEEL. Para isso, o sistema utiliza diferentes ferramentas de NLP, Word Sense Disambiguation (WSD), embeddings semânticos e clustering automático.

## Ferramentas utilizadas

- spaCy
- NLTK
- WordNet
- GlossBERT
- Sentence-BERT
- HDBSCAN
- PyTorch
- Transformers (Hugging Face)

---

# OBJETIVO

O sistema identifica stems frequentes e os sentidos semânticos mais prováveis das palavras derivadas desses stems em um corpus textual. Além disso, o pipeline:

- valida ambiguidades lexicais via Word Sense Disambiguation (WSD);
- gera definições semânticas contextualizadas;
- cria clusters semânticos concisos;
- permite revisão manual de definições e agrupamentos;
- exporta os resultados estruturados para JSON e HTML.

---

# REQUISITOS

## Arquivos necessários

- Corpus textual `.txt`
- Pasta local:

```text
./GlossBERT_Checkpoint
```

## A pasta `GlossBERT_Checkpoint` deve conter:

- `config.json`
- `pytorch_model.bin`
- `vocab.txt`
- `tokenizer_config.json`
- demais arquivos do checkpoint do modelo

---

# DEPENDÊNCIAS

```bash
pip install torch spacy nltk transformers sentence-transformers hdbscan numpy tqdm
```

---

# RECURSOS NLTK NECESSÁRIOS

```python
nltk.download("wordnet")
nltk.download("omw-1.4")
```

---

# RECURSO SPACY NECESSÁRIO

```python
python -m spacy download en_core_web_sm
```

---
# FLUXO GERAL

1. Processamento linguístico do corpus  
2. Extração de stems frequentes  
3. Mapeamento:
   - palavra
   - stem
   - POS
   - sentença
4. Recuperação de synsets WordNet  
5. Predição semântica com GlossBERT  
6. Revisão manual opcional  
7. Geração de definições aceitas  
8. Criação de embeddings semânticos (Sentence-BERT)  
9. Clustering automático (HDBSCAN)  
10. Revisão manual dos clusters  
11. Exportação final para JSON e HTML  

---

# OBSERVAÇÕES

- Alguns stems podem não existir no WordNet
- Clusters considerados ruído são descartados automaticamente
- Execução com GPU é recomendada para melhor desempenho
- O sistema permite revisão manual tanto das definições quanto dos clusters semânticos
- O pipeline foi projetado para apoiar workflows interpretativos do PEEL localmente

In [ ]:
# IMPORTS
from collections import Counter, defaultdict
import math
import torch
import spacy
import nltk
from nltk.stem import PorterStemmer
from tqdm.auto import tqdm
from transformers import (BertTokenizer, BertForSequenceClassification)
from nltk.corpus import wordnet as wn
from sentence_transformers import SentenceTransformer
import hdbscan
import numpy as np
import json
import re

# REQUIRED DOWNLOADS
nltk.download("wordnet")
nltk.download("omw-1.4")

# REQUIRED UPLOADS
# Upload the .txt file you wish to analyze and
# also the GlossBERT_Checkpoint folder
# Find GlossBERT_Checkpoint here: https://github.com/HSLCY/GlossBERT

In [ ]:
# DEVICE
# This pipeline requires a GPU or else it'll take a long time or not even run at all. Choose an environment with a GPU and check if the output here is "cuda"
# If output is "cuda", proceed

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"\nUsing device: {device}")

In [ ]:
# FUNCTION TO AUTOMATICALLY NAME A CORPUS:

def corpus_name(text):
  match = re.match(r"^[^.]+", text)
  if match:
    result = match.group(0)
    return(result)

In [ ]:
# GLOBAL VARIABLES AND MODELS

# VARIABLES
TXT_FILE = "Boisseau.txt"      # Path to your .txt file, here I already included an example
CORPUS_NAME = corpus_name(TXT_FILE)
TOP_PERCENTILE = 0.50       # Example: top 50% most frequent stems
MAX_STEMS = 150             # Maximum number of stems to return
TOP_STEMS = "top_stems.txt" # File with top stems
GLOSSBERT_OUTPUT = "glossbert_accepted_terms.txt" # File with GlossBERT results
MAX_SENTENCES_PER_STEM = 5 # For each collected stems, get N sentences that contain it
MAX_SYNSETS = 5 # For each stem, get 5 WordNet definitions of terms represented by it
FINAL_JSON = f"{CORPUS_NAME}-phase1_state.json"
HTML_PATH = f"{CORPUS_NAME}-Phase1-clusters.html" # Tableau20 HTML



# MODELS
LANG_MODEL = "en_core_web_sm" # spaCy's language model
nlp = spacy.load(LANG_MODEL)
stemmer = PorterStemmer() # Stemmer from NLTK, spaCy does not work with stems (only lemmas)
MODEL_PATH = "./GlossBERT_Checkpoint" # GlossBERT to perform definition-checking steps, find it here: https://github.com/HSLCY/GlossBERT
sentence_embedder = "all-MiniLM-L6-v2" # Model to be used to generate sentence embeddings for semantic clustering

In [ ]:
# READ TEXT FILE

with open(TXT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

# PROCESS TEXT

doc = nlp(text)


# EXTRACT STEMS USING NLTK


tokens = []

for token in doc:
    if (
        not token.is_stop and      # remove stopwords
        not token.is_punct and     # remove punctuation
        not token.is_space and     # remove spaces
        token.is_alpha             # keep alphabetic tokens only
    ):
        stem = stemmer.stem(token.text.lower())
        tokens.append(stem)

# COUNT FREQUENCIES

freq = Counter(tokens)

# SORT BY FREQUENCY DESCENDING
sorted_freq = sorted(freq.items(), key=lambda x: x[1], reverse=True)

# APPLY PERCENTILE FILTER

percentile_n = math.ceil(len(sorted_freq) * TOP_PERCENTILE)

final_n = min(percentile_n, MAX_STEMS) # Respect both percentile and absolute maximum

top_stems = dict(sorted_freq[:final_n]) # Final selection

# OUTPUT

print(f"\nTotal unique stems: {len(sorted_freq)}")
print(f"Returning top {final_n} stems\n")

for stem, count in top_stems.items():
    print(f"{stem}: {count}")


# SAVE TO FILE OS HASH (#) IF UNDESIRED


with open(TOP_STEMS, "w", encoding="utf-8") as out:
    out.write("stem\tcount\n")
    for stem, count in top_stems.items():
        out.write(f"{stem}\t{count}\n")

print(f"\nSaved results to {TOP_STEMS}")

In [ ]:
# MAPPING OF SPACY'S POS CATEGORIES ONTO NLTK
# This code uses WordNet, downloaded via NLTK, for word definitions, but spaCy's POS tagger is more up-to-date
# Therefore, I map spaCy's tags onto NLTK for POS representations

POS_MAP = {

    "NOUN": wn.NOUN,

    "VERB": wn.VERB,

    "ADJ": wn.ADJ,

    "ADV": wn.ADV
}

# LOAD GLOSSBERT CHECKPOINT

print("\nLoading GlossBERT checkpoint...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_PATH
)

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)

model.eval()

print("GlossBERT loaded successfully.")

# EXTRACTION OF SENTENCES THAT CONTAIN THE RELEVANT STEMS

print("\nMapping stems to sentences...")

stem_occurrences = defaultdict(list)

sentences_list = list(doc.sents)

print("\nMapping stem occurrences...")

sentences_list = list(doc.sents)

for sent in tqdm(
    sentences_list,
    desc="Sentence mapping"
):

    sent_text = sent.text.strip()

    for token in sent:

        if not token.is_alpha:
            continue

        stem = stemmer.stem(token.text.lower())

        if stem not in top_stems:
            continue

        occurrence = {

            # Actual surface word
            "word": token.text,

            # Stem
            "stem": stem,

            # POS tag
            "pos": token.pos_,

            # Sentence
            "sentence": sent_text,

            # Character positions
            "start": token.idx,
            "end": token.idx + len(token.text)
        }

        stem_occurrences[stem].append(
            occurrence
        )

# GLOSSBERT FUNCTION

def glossbert_predict(occurrence):

    word = occurrence["word"]

    stem = occurrence["stem"]

    pos = occurrence["pos"]

    sentence = occurrence["sentence"]

    wn_pos = POS_MAP.get(pos)

    # POS-filtered synsets using WORD

    synsets = wn.synsets(
        word,
        pos=wn_pos
    )

    if len(synsets) == 0:
        return None

    synsets = synsets[:MAX_SYNSETS]

    results = []

    # Mark word for classification

    marked_sentence = sentence.replace(
        word,
        f"[TGT] {word} [TGT]",
        1
    )

    for syn in synsets:

        gloss = syn.definition()

        encoding = tokenizer(
            marked_sentence,
            gloss,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length"
        )

        input_ids = encoding[
            "input_ids"
        ].to(device)

        attention_mask = encoding[
            "attention_mask"
        ].to(device)

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            probs = torch.softmax(
                logits,
                dim=1
            )

            match_score = probs[
                0
            ][1].item()

        results.append({

            "synset": syn,

            "definition": gloss,

            "score": match_score
        })

    results = sorted(
        results,
        key=lambda x: x["score"],
        reverse=True
    )

    return results

# GLOSSBERT RESULTS

print("\nRunning GlossBERT analysis...")

accepted_definitions = {}

flagged_words = []

print("\nRunning GlossBERT analysis...")

accepted_definitions = {}

flagged_words = []

for stem, count in tqdm(
    top_stems.items(),
    total=len(top_stems),
    desc="GlossBERT"
):

    occurrences = stem_occurrences[stem]

    if len(occurrences) == 0:
        continue

    occurrences = occurrences[
        :MAX_SENTENCES_PER_STEM
    ] # Limit max sentences per stem for saving computational demand

    mismatch_found = False
    for occurrence in occurrences:

        word = occurrence[
            "word"
        ].lower()

        pos = occurrence["pos"]

        wn_pos = POS_MAP.get(pos)

        # WORD-LEVEL SYNSETS


        synsets = wn.synsets(
            word,
            pos=wn_pos
        )

        # SKIP IF NO SYNSETS

        if len(synsets) == 0:
            continue

        synsets = synsets[
            :MAX_SYNSETS
        ]

        # DEFAULT SENSE FOR WORD

        default_sense = synsets[0]

        # GLOSSBERT PREDICTION

        results = glossbert_predict(
            occurrence
        )

        if results is None:
            continue

        best_sense = results[0]["synset"]

        # FLAG MISMATCHES

        if (
            best_sense.name()
            != default_sense.name()
        ):

            mismatch_found = True
            #TODO think about other ways of computing mismatches if mismatches list is constantly large

            flagged_words.append({

                "word":
                    occurrence["word"],
                "stem":
                    stem,
                "pos":
                    pos,
                "count":
                    count,
                "sentence":
                    occurrence["sentence"],
                "default_sense":
                    default_sense.name(),
                "default_definition":
                    default_sense.definition(),
                "predicted_sense":
                    best_sense.name(),
                "predicted_definition":
                    best_sense.definition(),
                "top_candidates": [

                    {
                        "sense":
                            r["synset"].name(),

                        "definition":
                            r["definition"],

                        "score":
                            round(
                                r["score"],
                                4
                            )
                    }

                    for r in results[:3]
                ]
            })

            break

    # ACCEPT DEFAULT SENSE

    if not mismatch_found:

        first_occurrence = occurrences[0]

        word = first_occurrence[
            "word"
        ].lower()

        pos = first_occurrence["pos"]

        wn_pos = POS_MAP.get(pos)

        synsets = wn.synsets(
            word,
            pos=wn_pos
        )

        if len(synsets) == 0:
            continue

        accepted_definitions[stem] = {

            "word":
                first_occurrence["word"],

            "pos":
                pos,

            "count":
                count,

            "definition":
                synsets[0].definition()
        }

# OUTPUT RESULTS

print("\n===================================")
print("ACCEPTED DEFINITIONS")
print("===================================\n")

for stem, data in accepted_definitions.items():

    print({
        "word": data["word"],
        "stem": stem,
        "pos": data["pos"],
        "count": data["count"],
        "definition": data["definition"]})

# FLAGGED TERMS

print("\n===================================")
print("FLAGGED TERMS")
print("===================================\n")

for item in flagged_words:

    print("\n-----------------------------------")

    print(f"WORD: {item['word']}")

    print(f"STEM: {item['stem']}")

    print(f"POS: {item['pos']}")

    print(f"COUNT: {item['count']}")

    print("\nSENTENCE:")

    print(item["sentence"])

    print("\nDEFAULT SENSE:")

    print(item["default_sense"])

    print(item["default_definition"])

    print("\nPREDICTED SENSE:")

    print(item["predicted_sense"])

    print(item["predicted_definition"])

    print("\nTOP 3 CANDIDATES:")

    for candidate in item["top_candidates"]:

        print(
            f"- {candidate['sense']}"
        )

        print(
            f"  DEF: "
            f"{candidate['definition']}"
        )

        print(
            f"  SCORE: "
            f"{candidate['score']}"
        )


print("\nDone.")

In [ ]:
# USER REVIEW OF FLAGGED TERMS

print("\n===================================")
print("FLAGGED TERM REVIEW")
print("===================================\n")

if len(flagged_words) == 0:

    print("No flagged terms found.")

else:

    # ASK USER IF THEY WANT TO ACCEPT ALL

    accept_all = input(
        "\nAccept all predicted definitions "
        "from flagged terms? (y/n): "
    ).strip().lower()

    # ACCEPT ALL

    if accept_all == "y":

        for item in flagged_words:

            accepted_definitions[
                item["stem"]
            ] = {

                "word":
                    item["word"],

                "pos":
                    item["pos"],

                "count":
                    item["count"],

                "definition":
                    item["predicted_definition"]
            }

        print(
            "\nAll predicted definitions accepted."
        )

    # MANUAL REVIEW

    else:

        review_mode = input(

            "\nReview flagged terms:\n"
            "[1] One by one\n"
            "[2] Select from all terms\n\n"
            "Choice: "

        ).strip()

        # FUNCTION TO REVIEW ONE ITEM

        def review_flagged_item(item):

            print("\n===================================")

            print(f"WORD: {item['word']}")

            print(f"STEM: {item['stem']}")

            print(f"POS: {item['pos']}")

            print(f"COUNT: {item['count']}")

            print("\nSENTENCE:")

            print(item["sentence"])

            print("\nDEFAULT SENSE:")

            print(
                f"{item['default_sense']}"
            )

            print(
                f"{item['default_definition']}"
            )

            print("\nPREDICTED SENSE:")

            print(
                f"{item['predicted_sense']}"
            )

            print(
                f"{item['predicted_definition']}"
            )

            print("\nTOP CANDIDATES:\n")


            # PRINT CANDIDATES FOR UPDATED DEFINITION

            for idx, candidate in enumerate(
                item["top_candidates"],
                start=1
            ):

                print(
                    f"[{idx}] "
                    f"{candidate['sense']}"
                )

                print(
                    f"DEF: "
                    f"{candidate['definition']}"
                )

                print(
                    f"SCORE: "
                    f"{candidate['score']}\n"
                )

            # USER CHOICE PART

            print(
                "[0] Keep default definition"
            )

            print(
                "[4] Enter manual definition"
            )

            choice = input(
                "\nChoice: "
            ).strip()

            # IF KEEP DEFAULT

            if choice == "0":

                chosen_definition = (
                    item["default_definition"]
                )

            # MANUAL DEFINITION

            elif choice == "4":

                chosen_definition = input(
                    "\nEnter custom definition: "
                ).strip()

            # CHOOSE FROM TOP CANDIDATES

            elif choice in ["1", "2", "3"]:

                candidate = item[
                    "top_candidates"
                ][int(choice) - 1]

                chosen_definition = (
                    candidate["definition"]
                )

            # IF INVALID INPUT

            else:

                print(
                    "\nInvalid option."
                )

                return

            # UPDATE ACCEPTED DEFINITIONS

            accepted_definitions[
                item["stem"]
            ] = {

                "word":
                    item["word"],

                "pos":
                    item["pos"],

                "count":
                    item["count"],

                "definition":
                    chosen_definition
            }

            print(
                "\nDefinition updated."
            )

        # ONE-BY-ONE REVIEW

        if review_mode == "1":

            for item in flagged_words:

                review_flagged_item(item)

        # SELECTIVE REVIEW

        elif review_mode == "2":

            print(
                "\nFLAGGED TERMS:\n"
            )

            for item in flagged_words:

                print(
                    f"- {item['word']} "
                    f"(stem={item['stem']})"
                )

            while True:

                selected_word = input(

                    "\nType a word to review "
                    "(or 'exit'): "

                ).strip()

                if selected_word.lower() == "exit":
                    break

                found = False

                for item in flagged_words:

                    if (
                        item["word"].lower()
                        == selected_word.lower()
                    ):

                        review_flagged_item(item)

                        found = True

                        break

                if not found:

                    print(
                        "\nWord not found."
                    )

# SAVE UPDATED RESULTS

print("\nSaving updated results...")

with open(
    GLOSSBERT_OUTPUT,
    "w",
    encoding="utf-8"
) as out:

    out.write(
        "word\tstem\tpos\tcount\tdefinition\n"
    )

    for stem, data in accepted_definitions.items():

        out.write(
            f"{data['word']}\t"
            f"{stem}\t"
            f"{data['pos']}\t"
            f"{data['count']}\t"
            f"{data['definition']}\n"
        )

print(
    f"\nUpdated results saved to "
    f"{GLOSSBERT_OUTPUT}"
)

In [ ]:
# LOAD SENTENCE-BERT

embedder = SentenceTransformer(
    sentence_embedder
)

# BUILD TEXT REPRESENTATIONS BY VECTORIZING SENTENCES

stem_texts = []
stem_names = []

for stem, data in accepted_definitions.items():

    definition = data["definition"]

    text_representation = (
        f"{stem}: {definition}"
    )

    stem_texts.append(
        text_representation
    )

    stem_names.append(
        stem
    )

# CREATE EMBEDDINGS

embeddings = embedder.encode(
    stem_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# HDBSCAN CLUSTERING

clusterer = hdbscan.HDBSCAN(

    min_cluster_size=3,

    metric="euclidean",

    cluster_selection_method="eom"
)

labels = clusterer.fit_predict(
    embeddings
)

# ORGANIZE CLUSTERS

clusters = {}

for stem, label in zip(
    stem_names,
    labels
):

    # Ignore noise (all -1 categories are non-confident clusters)

    if label == -1:
        continue

    if label not in clusters:
        clusters[label] = []

    clusters[label].append(
        stem
    )

# CLUSTERS ARE PRINTED IN THE NEXT CELL

In [ ]:
# CLUSTER RENAMING
# This cell composes the cluster-labeling step. It's characterized by a simple naming process
# That chooses cluster titles by frequency

stem_word_frequencies = {}

for stem, occurrences in stem_occurrences.items():

    words = [

        occ["word"].lower()

        for occ in occurrences
    ]

    stem_word_frequencies[stem] = Counter(words)

# NAME CLUSTERS BASED ON FREQUENCY OF TOP-OCCURRING WORD(S)

renamed_clusters = {}

for label, stems in clusters.items():

    # Aggregate word frequencies

    cluster_word_counter = Counter()

    for stem in stems:

        if stem in stem_word_frequencies:

            cluster_word_counter.update(
                stem_word_frequencies[stem]
            )

    # Skip empty clusters

    if len(cluster_word_counter) == 0:
        continue

    # Find highest frequency

    max_freq = max(
        cluster_word_counter.values()
    )

    # Get all tied words

    top_words = sorted([

        word

        for word, freq in
        cluster_word_counter.items()

        if freq == max_freq
    ])

    # Build cluster name

    cluster_name = " & ".join(
        top_words
    ) # Words are joined if tied for highest frequency

    renamed_clusters[
        cluster_name
    ] = {

        "label": label,

        "stems": stems,

        "top_frequency": max_freq,

        "top_words": top_words
    }

# SEMANTIC CLUSTERS

print("\n======================")
print("SEMANTIC CLUSTERS")
print("======================\n")

for cluster_name in sorted(
    renamed_clusters.keys()
):

    cluster_data = renamed_clusters[
        cluster_name
    ]

    print(
        f"CLUSTER: {cluster_name}"
    )

    print(
        "STEMS:"
    )

    print(
        ", ".join(
            cluster_data["stems"]
        )
    )

    print()

In [ ]:
# Additionally, users may choose to rename clusters
# INTERACTIVE CLUSTER REVIEW

print("\n======================")
print("CLUSTER REVIEW")
print("======================\n")

final_clusters = {}


# Track excluded stems if the user chooses to exclude any

excluded_cluster_stems = set()

for cluster_name in sorted(
    renamed_clusters.keys()
):

    cluster_data = renamed_clusters[
        cluster_name
    ]

    stems = cluster_data["stems"]

    print("\n----------------------------------")
    print(f"CLUSTER NAME: {cluster_name}")

    print("\nSTEMS:")

    for i, stem in enumerate(stems):

        print(f"{i+1}. {stem}")

    print("\nAccept this cluster?")

    accept = input(
        "(y = accept / n = modify): "
    ).strip().lower()


    # ACCEPT AS-IS

    if accept == "y":

        final_clusters[cluster_name] = {

            "stems": stems,

            "excluded_stems": []
        }

        continue

    # CHANGE CLUSTER NAME

    new_name = input(
        "\nNew cluster name "
        "(leave empty to keep current): "
    ).strip()

    if new_name == "":

        new_name = cluster_name


    # REMOVE STEMS

    print("\nCurrent stems:")

    for i, stem in enumerate(stems):

        print(f"{i+1}. {stem}")

    remove_input = input(
        "\nType stem numbers to remove "
        "\n(comma-separated and in any order as in '5,2,3,9...') "
        "\n or press ENTER to keep all: "
    ).strip()

    updated_stems = stems.copy()

    removed_stems = []

    if remove_input != "":

        try:

            remove_indices = [

                int(x.strip()) - 1

                for x in remove_input.split(",")
            ]

            removed_stems = [

                stem

                for i, stem in enumerate(stems)

                if i in remove_indices
            ]

            updated_stems = [

                stem

                for i, stem in enumerate(stems)

                if i not in remove_indices
            ]

            excluded_cluster_stems.update(
                removed_stems
            )

        except:

            print(
                "\nInvalid input."
            )

            print(
                "Keeping all stems."
            )

    # SAVE UPDATED CLUSTER

    final_clusters[new_name] = {

        "stems": updated_stems,

        "excluded_stems": removed_stems
    }

# FINAL OUTPUT

print("\n======================")
print("FINAL CLUSTERS")
print("======================\n")

for cluster_name in sorted(
    final_clusters.keys()
):

    print(
        f"CLUSTER: {cluster_name}"
    )

    print(
        "STEMS:"
    )

    print(
        ", ".join(
            final_clusters[
                cluster_name
            ]["stems"]
        )
    )

    excluded = final_clusters[
        cluster_name
    ]["excluded_stems"]

    if len(excluded) > 0:

        print(
            "EXCLUDED:"
        )

        print(
            ", ".join(excluded)
        )

    print()

In [ ]:
# SAVE JSON

phase1_state = {

    # Included stems

    "incList": sorted(

        list({

            stem

            for cluster_data in final_clusters.values()

            for stem in cluster_data["stems"]
        })
    ),

    # Excluded stems

    "excList": sorted(
        list(excluded_cluster_stems)
    ),

    # Cluster definitions

    "clusterDefs": [

        {

            # Cluster name
            "name": cluster_name,

            # Final stems
            "stems": final_clusters[
                cluster_name
            ]["stems"],

            # Removed stems
            "excluded_stems": final_clusters[
                cluster_name
            ]["excluded_stems"]

        }

        for cluster_name in sorted(
            final_clusters.keys()
        )
    ]
}

# SAVE JSON

with open(
    FINAL_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        phase1_state,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"\nSaved JSON to {FINAL_JSON}")

In [ ]:
# HTML CLUSTER EXPORT

TABLEAU20 = [
    "#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F",
    "#EDC948","#B07AA1","#FF9DA7","#9C755F","#BAB0AC",
    "#499894","#A0CBE8","#FFBE7D","#FF9D9A","#86BCB6",
    "#8CD17D","#F1CE63","#D4A6C8","#FABFD2","#D7B5A6",
]


# HEX TO RGB
def hex_to_rgb(h):

    h = h.lstrip('#')

    return tuple(
        int(h[i:i+2], 16)

        for i in (0, 2, 4)
    )

# BUILD TABLE ROWS

rows = []

# ------------------------------------------
# final_clusters is now a dict:
#
# {
#     cluster_name: {
#         "stems": [...],
#         "excluded_stems": [...]
#     }
# }
# ------------------------------------------

for i, cluster_name in enumerate(
    sorted(final_clusters.keys())
):

    cluster_data = final_clusters[
        cluster_name
    ]

    stems = cluster_data["stems"]

    excluded = cluster_data[
        "excluded_stems"
    ]

    color = TABLEAU20[
        i % len(TABLEAU20)
    ]

    r, g, b = hex_to_rgb(color)

    # Included stems

    stems_str = ", ".join(

        f'<code>{s}</code>'

        for s in stems
    )

    # Excluded stems

    excluded_html = ""

    if len(excluded) > 0:

        excluded_str = ", ".join(

            f'<code>{s}</code>'

            for s in excluded
        )

        excluded_html = (
            f'<div style="margin-top:4px; '
            f'font-size:0.8em; '
            f'color:#999;">'
            f'Excluded: {excluded_str}'
            f'</div>'
        )

    # Row

    rows.append(

        f'    <tr>\n'

        f'      <td style="padding:5px 12px 5px 0;">'
        f'&nbsp;</td>\n'

        f'      <td style="padding:5px 12px 5px 0; '
        f'color:rgb({r},{g},{b}); '
        f'font-weight:bold;">'
        f'{cluster_name}'
        f'</td>\n'

        f'      <td style="padding:5px 0; '
        f'font-size:0.88em; '
        f'color:#555;">'
        f'{stems_str}'
        f'{excluded_html}'
        f'</td>\n'

        f'    </tr>'
    )

# JOIN ROWS

rows_html = '\n'.join(rows)

# TOTAL STEMS

total_stems = sum(

    len(cluster_data["stems"])

    for cluster_data in final_clusters.values()
)

# HTML SNIPPET

snippet = f"""
<h3>Semantic Clusters — Phase 1 results</h3>

<p style="font-style:italic;
          color:#666;
          font-size:0.9em;">

  {corpus_name} &mdash;
  {len(final_clusters)} clusters &middot;
  {total_stems} stems &middot;
  Tableau20 palette

</p>

<table style="border-collapse:collapse;
              font-family:serif;
              font-size:14px;">

  <thead>

    <tr>

      <th style="padding:5px 12px 5px 0;">
        &nbsp;
      </th>

      <th style="padding:5px 12px 5px 0;
                 text-align:left;">
        Cluster
      </th>

      <th style="padding:5px 0;
                 text-align:left;">
        Stems
      </th>

    </tr>

  </thead>

  <tbody>

{rows_html}

  </tbody>

</table>
"""

# SAVE HTML

with open(
    HTML_PATH,
    'w',
    encoding='utf-8'
) as f:

    f.write(snippet)

print(
    f"\nHTML cluster snippet written:\n"
    f"{HTML_PATH}"
)